In [9]:
import pandas as pd
win = pd.read_csv("windows.csv")
cols = ["ax", "ay", "az", "gx", "gy", "gz"]
feat = [c + "_std" for c in cols]
X = win[feat]
y = win["label"]
groups = win["ride_id"]
print(X.shape, y.shape, groups.nunique())

(10714, 6) (10714,) 36


In [10]:
from sklearn.model_selection import GroupKFold
gkf = GroupKFold(n_splits= 4)
for train_id, test_id in gkf.split(X, y, groups):
    print(len(train_id), len(test_id), groups.iloc[test_id].nunique())

8035 2679 9
8035 2679 9
8036 2678 9
8036 2678 9


In [11]:
for train_id, test_id in gkf.split(X, y, groups):
    print(y.iloc[test_id].value_counts().to_dict())

{'normal': 1787, 'calm': 595, 'aggressive': 297}
{'calm': 1192, 'normal': 892, 'aggressive': 595}
{'calm': 1190, 'aggressive': 892, 'normal': 596}
{'normal': 1487, 'calm': 596, 'aggressive': 595}


In [12]:
for train_id, test_id in gkf.split(X, y, groups):
    print(sorted(groups.iloc[test_id].unique()))

['R003', 'R004', 'R010', 'R013', 'R017', 'R022', 'R026', 'R028', 'R036']
['R001', 'R002', 'R007', 'R011', 'R016', 'R021', 'R024', 'R027', 'R033']
['R006', 'R009', 'R014', 'R019', 'R020', 'R025', 'R030', 'R032', 'R035']
['R005', 'R008', 'R012', 'R015', 'R018', 'R023', 'R029', 'R031', 'R034']


here the work of groupkfold is to isolate a certain group to either test or train at a time , a group R001 as eg can't be both in train and test at the same it , there would be data leakage.. and there is also no repitition of the groups.


In [13]:
for train_id, test_id in gkf.split(X, y, groups):
    print(y.iloc[test_id].value_counts().to_dict())

{'normal': 1787, 'calm': 595, 'aggressive': 297}
{'calm': 1192, 'normal': 892, 'aggressive': 595}
{'calm': 1190, 'aggressive': 892, 'normal': 596}
{'normal': 1487, 'calm': 596, 'aggressive': 595}


The issue of just GroupKFold is that the distribution of the groups is uneven mixture of the rides, so the scores aren't comparable
fix: we use stratifiedGroupKFold -> same as GroupKFold but better mix

In [14]:
from sklearn.model_selection import StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=0)
for train_idx, test_idx in sgkf.split(X, y, groups):
    print(y.iloc[test_idx].value_counts().to_dict())

{'normal': 1191, 'calm': 893, 'aggressive': 595}
{'normal': 1190, 'calm': 894, 'aggressive': 595}
{'normal': 1190, 'calm': 893, 'aggressive': 595}
{'normal': 1191, 'calm': 893, 'aggressive': 594}


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

pipe = Pipeline([("scale", StandardScaler()),
                 ("lr", LogisticRegression(max_iter=1000))])
scores = cross_val_score(pipe, X, y, groups=groups, cv=sgkf, scoring="f1_macro")
print(scores, scores.mean())

[0.44581762 0.59117155 0.42064497 0.50066659] 0.4895751808830424


In [ ]:
from sklearn.model_selection import cross_val_predict
pred = cross_val_predict(pipe, X, y, groups =groups, cv = sgkf)
print(pd.crosstab(y, pred)) # easier method to print confusion matrixx

col_0       aggressive  calm  normal
label                               
aggressive         678   314    1387
calm                99  2016    1458
normal             448  1417    2897
